# Check24 Search Routing with Clef

This notebook tests Cloudflare's Clef decision models through Ollama's local `/v1/systemone` API. It simulates the Check24 Search Team's hot-path query router: a user query goes in as `state`, and the model returns a probability distribution over verticals in a single non-autoregressive pass.

Requires Ollama 0.35.1 or later and a pulled model (`ollama pull clef:27b`).

`clef-flash` is the faster variant meant for the hot path, but on Ollama 0.35.1 it fails on `/v1/systemone` with "Clef: non-finite logit" ([ollama/ollama#18769](https://github.com/ollama/ollama/issues/18769)), so the notebook uses `clef:27b` until that is fixed.

In [9]:
import time

import requests

OLLAMA_API_URL = "http://ollama:11434/v1/systemone"
MODEL = "tev1:0.8b"

### 1. Define the Decision Schema

We ask two questions in one request:
1. **`target_vertical` (`choice`)**: Which Check24 product vertical best matches the query? The `other` option keeps unrelated queries from being forced onto a vertical.
2. **`requires_financing` (`noul`)**: Does the query indicate the user needs a loan or credit (useful for cross-selling or pre-applying facets)?

Every question needs `type` and `instructions`; `choice` also needs `criteria` mapping each option to a description.

In [10]:
routing_questions = {
    "target_vertical": {
        "type": "choice",
        "instructions": "Which Check24 product vertical best matches the search query?",
        "criteria": {
            "strom_gas": "Stromvergleich, Gasanbieter, Energie",
            "kfz_versicherung": "Autoversicherung, Motorradversicherung, Kfz-Tarife",
            "kredit": "Ratenkredit, Baufinanzierung, Autokredit",
            "reisen": "Pauschalreisen, Urlaub, Mietwagen, Flüge",
            "dsl_internet": "Internet, WLAN, Festnetz, Breitband",
            "other": "None of the above",
        },
    },
    "requires_financing": {
        "type": "noul",
        "instructions": "Does the user explicitly mention needing a loan, credit, borrowing money, or financing a purchase?",
    },
}

### 2. Create the Evaluation Function
The query is sent as the `state`. The response holds one entry per question under `answers`.

`confidence` measures how concentrated the probabilities are, not the chance that the answer is right, so routing thresholds should use `probabilities`.

In [11]:
def ask(user_query):
    """Return the model's answers and the request latency in milliseconds."""
    payload = {"model": MODEL, "state": user_query, "questions": routing_questions}
    start = time.perf_counter()
    response = requests.post(OLLAMA_API_URL, json=payload, timeout=120)
    latency_ms = (time.perf_counter() - start) * 1000
    response.raise_for_status()
    return response.json()["answers"], latency_ms


def route_query(user_query):
    """Print the routing decision for one query."""
    answers, latency_ms = ask(user_query)
    vertical = answers["target_vertical"]

    print(f"Query: '{user_query}'")
    print(f"Latency: {latency_ms:.2f} ms\n")

    print("--- Routing Decision ---")
    print(f"Top Vertical: {vertical['choice']} (concentration: {vertical['confidence']:.4f})")

    print("\n--- Probability Distribution (for hybrid search) ---")
    for category, prob in sorted(vertical["probabilities"].items(), key=lambda x: x[1], reverse=True):
        print(f"{category:<20}: {prob:.4f}")

    print("\n--- Facet Extraction ---")
    print(f"Requires Financing (probability): {answers['requires_financing']['noul']:.4f}")
    print("=" * 60 + "\n")

### 3. Run Test Queries
Let's test both clear intents and ambiguous Check24 scenarios (like buying a car with a loan).

The first call loads the model into memory, so we warm up once to keep load time out of the latency numbers.

In [12]:
test_queries = [
    "Ich brauche einen neuen Router für meine Wohnung, am besten Glasfaser.",
    "Günstiger Tarif für mein neues E-Auto inkl. Vollkasko",
    "Ich möchte mir einen Gebrauchtwagen kaufen, brauche aber 15.000 Euro dafür",
    "All inclusive 2 Wochen Mallorca für 2 Personen",
]

ask("warm-up")

for q in test_queries:
    route_query(q)

Query: 'Ich brauche einen neuen Router für meine Wohnung, am besten Glasfaser.'
Latency: 693.42 ms

--- Routing Decision ---
Top Vertical: dsl_internet (concentration: 0.7854)

--- Probability Distribution (for hybrid search) ---
dsl_internet        : 0.9006
other               : 0.0822
reisen              : 0.0122
strom_gas           : 0.0033
kfz_versicherung    : 0.0010
kredit              : 0.0007

--- Facet Extraction ---
Requires Financing (probability): 0.0184

Query: 'Günstiger Tarif für mein neues E-Auto inkl. Vollkasko'
Latency: 693.26 ms

--- Routing Decision ---
Top Vertical: kfz_versicherung (concentration: 0.9344)

--- Probability Distribution (for hybrid search) ---
kfz_versicherung    : 0.9768
other               : 0.0217
strom_gas           : 0.0006
kredit              : 0.0004
reisen              : 0.0003
dsl_internet        : 0.0002

--- Facet Extraction ---
Requires Financing (probability): 0.6398

Query: 'Ich möchte mir einen Gebrauchtwagen kaufen, brauche aber 15.0